# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 25 · Frozen training-fit audit

**No optimizer is created.** Forward-replay both completed models on every selected training row. The historical evaluation metrics are read from the existing summary; evaluation arrays are not opened. Training error is in-sample and cannot validate a feature. Earlier online training objectives are not final-checkpoint errors.

In [ ]:
from pathlib import Path
import json, sys, subprocess, signal
import plotly.io as pio
KIT = Path('/home/sagemaker-user/nfl_feature_round11')
OUT = Path('/home/sagemaker-user/nfl-feature-round11-results')
PY = Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Use the existing NFL space and extract this kit first.')
sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'
def run(stage):
    p = subprocess.Popen([str(PY),str(KIT/'run_round.py'),stage], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,bufsize=1)
    try:
        for line in p.stdout: print(line,end='')
        code=p.wait()
    except KeyboardInterrupt:
        p.send_signal(signal.SIGINT)
        try:p.wait(timeout=10)
        except subprocess.TimeoutExpired:p.kill();p.wait()
        raise
    if code:
        raise RuntimeError(f'{stage} stopped ({code}). Export the report; do not loosen gates or reinstall packages.')
def show(fig,name):
    visuals.save(fig,OUT,name).show()


## 1. Audit the saved models
Uses the existing offline CPU runtime and first runs eight synthetic model/metric tests. It must reproduce the original first-eight-play probes before continuing. New candidate features do not enter the saved models.

In [ ]:
run('audit')
r=json.loads((OUT/'training_audit.json').read_text())
print(json.dumps({'training_rows':r['training_rows'],'metrics':r['metrics'],'new_optimizer_steps':r['new_optimizer_steps']},indent=2))

In [ ]:
show(visuals.training_vs_evaluation(OUT),'fit_vs_evaluation')
show(visuals.horizon_errors(OUT),'training_horizon_errors')
show(visuals.learning(KIT),'recorded_learning')

## 2. Recompute in a fresh process
Rebuild candidate features and recompute training predictions exactly. Missing prediction/feature checkpoints are rejected. No fit, new holdout score, ensemble, or model promotion is permitted.

In [ ]:
run('replay')
show(visuals.gates(OUT),'milestone_gates')

## 3. Export the aggregate report
Private per-play files stay here. Return only the report ZIP. A small training error with a larger evaluation error is consistent with a generalization problem, but does not prove the cause. High training error leaves representation, capacity and optimization unresolved. No numeric decision threshold is tuned to this audit.

In [ ]:
run('report')
print(OUT/'nfl_feature_round11_report.zip')

Save this notebook and stop the existing space when finished. No new fit is authorized by a successful audit alone.